In [12]:
import pandas as pd
import numpy as np
import plotly.graph_objs as go
from dash import Dash, dcc, html
from enum import Enum
import cv2
import json

video_ID = '84EXT'

vid = cv2.VideoCapture(f'/home/edisonz/maneuver_heuristics25/Parking-clip{video_ID}.mp4')
frame_width = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))  # Max x-coordinate
frame_height = 2*int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))//3  # Max y-coordinate
# ----------------------------------------------------------------------------------------------------------------------------

# Enter hyperparameters (adjustable)
start_entry_ratio = 0.5 # 0 = zone based | 1 = peak based 
end_entry_ratio = 1.0 # 0 = front parking zone | 1 = rear parking zone

# Exit hyperparameters (adjustable)
start_exit_ratio = 0 # 0 = rear parking zone | 1 = front parking zone
end_exit_ratio = 1.0 # 0 = peak based | 1 = zone based

# low motion hyperparameters 
movement_threshold = 1.0

# Peak hyperparameters 
tA = 5
aA = np.pi/6

def load_lines(json_file):
    with open(json_file, 'r') as f:
        data = json.load(f)
    lines = [list(map(tuple, line)) for line in data['lines']]
    loaded_lines = []
    for line in lines: 
        m = (line[1][1] - line[0][1])/(line[1][0] - line[0][0])
        c = line[0][1] - m * line[0][0]
        loaded_lines.append((m, c))
    return loaded_lines

# Load data
vehicle_df = pd.read_csv(f'clips/Parking-clip{video_ID}.csv')
maneuvers = pd.read_csv(f'clip-annotations/maneuver{video_ID}.csv')
start_frame, end_frame = np.median(maneuvers["start_frame"]), np.median(maneuvers["end_frame"])
driving_df = vehicle_df[vehicle_df["in_parking_zone"]==0]
driving_coords = driving_df[['x', 'y']].values
parking_df = vehicle_df[vehicle_df["in_parking_zone"]==1]
parking_coords = parking_df[['x', 'y']].values
start_frame = int(round(start_frame))
end_frame = int(round(end_frame))

region_code = None
with open("mappings.json", 'r') as f:
    data = json.load(f)
    region_code = data.get(f'{video_ID}')
parking_lines = f'lines{region_code}.json'

loaded_lines = load_lines(parking_lines)

class ManeuverType(Enum):
    ENT = 0
    EXT = 1

maneuver_type = ManeuverType.ENT if video_ID[-3:].lower()=='ent' else ManeuverType.EXT

# --------------------------------------------------------------------------------------------------------------------------

# Determine if direction reversal in x or y occured
def is_reversal(r1, r2):
    angle_diff = np.abs(np.arctan2(r1[1], r1[0]) - np.arctan2(r2[1], r2[0]))
    if angle_diff < aA:
        return False  # No significant angle change
    return (r1[0] * r2[0] < 0) or (r1[1] * r2[1] < 0)  # Opposite signs in x or y

# Determine if vehicle is hugging the border of the frame
def on_border(df_row):
    height = df_row["height"]
    width = df_row["width"]
    x = df_row["x"]
    y = df_row["y"]
    left_x = int(x-width//2)
    right_x = int(x+width//2)
    top_y = int(y-height//2)
    bottom_y = int(y+height//2)
    return (left_x==0 or top_y==0 or right_x==frame_width or bottom_y==frame_height)

def find_shortest_distance(point, loaded_lines):
    """
    Compute the shortest distance for a given point:
    - If the point lies between two parking lines, return the sum of perpendicular distances to the nearest lines on either side.
    - If the point does not lie between two lines, return the perpendicular distance to the closest line (min perp dist among all lines).
    Lines include both original from json and extrapolated ones.
    
    Args:
        point (tuple): (x, y) coordinates of the point
        json_file (str): Path to the JSON file containing line data
    
    Returns:
        float: Shortest distance (sum of perpendicular distances or min distance to closest line), or None if no lines exist
    """
    x_point, y_point = point

    # Original lines
    intersections = []
    for m, c in loaded_lines:
        x_inter = (y_point - c) / m
        intersections.append(x_inter)

    # Separate intersections into left and right
    left = [(x, params) for x, params in zip(intersections, loaded_lines) if x < x_point]
    right = [(x, params) for x, params in zip(intersections, loaded_lines) if x > x_point]

    # Compute perpendicular distance to a line
    def point_to_line_distance(point, line_params):
        x_p, y_p = point
        if line_params[1] is None:  # Vertical line: x = c
            return abs(x_p - line_params[0])
        m, c = line_params
        return abs(m * x_p - y_p + c) / np.sqrt(m**2 + 1)

    # Case 1: Point lies between two lines (considering all)
    if left and right:
        __, left_params = max(left, key=lambda x: x[0])
        __, right_params = min(right, key=lambda x: x[0])
        d_left = point_to_line_distance(point, left_params)
        d_right = point_to_line_distance(point, right_params)
        return d_left + d_right

    # Case 2: Point does not lie between two lines, find closest line by min perp dist
    if left and not right:
        __, left_params = max(left, key=lambda x: x[0])
        return point_to_line_distance(point, left_params)
    if right and not left:
        __, right_params = min(right, key=lambda x: x[0])
        return point_to_line_distance(point, right_params) 

def in_adjacent_zones(p1, p2, loaded_lines):
    """This function calculates whether two points are in the same or adjacent zones"""
    x_point1 = p1[0]
    y_point1 = p1[1]
    x_point2 = p2[0]
    y_point2 = p2[1]

    intersections1 = []
    intersections2 = []
    for m, c in loaded_lines:
        x_inter1 = (y_point1 - c) / m
        intersections1.append(x_inter1)
        x_inter2 = (y_point2 - c) / m
        intersections2.append(x_inter2)

    left1 = [(x, params) for x, params in zip(intersections1, loaded_lines) if x < x_point1]
    right1 = [(x, params) for x, params in zip(intersections1, loaded_lines) if x > x_point1]

    left2 = [(x, params) for x, params in zip(intersections2, loaded_lines) if x < x_point2]
    right2 = [(x, params) for x, params in zip(intersections2, loaded_lines) if x > x_point2]

    if len(left1) > 0: 
        __, left_params1 = max(left1, key = lambda x: x[0])  
    else: 
        left_params1 = None

    if len(right1) > 0:
        __, right_params1 = min(right1, key = lambda x: x[0]) 
    else:
        right_params1 = None

    if len(left2) > 0: 
        __, left_params2 = max(left2, key = lambda x: x[0])  
    else: 
        left_params2 = None

    if len(right2) > 0:
        __, right_params2 = min(right2, key = lambda x: x[0]) 
    else:
        right_params2 = None

    return (left_params1 == left_params2 or left_params1 == right_params2 or 
            right_params1 == right_params2 or right_params1 == left_params2)

if maneuver_type == ManeuverType.ENT:
    # --- End-of-Maneuver Detection (STOP) ---
    front_end_point = parking_coords[0]
    dists_from_front = np.linalg.norm(parking_coords - front_end_point, axis=1)

    rear_end_idx = np.argmax(dists_from_front)
    rear_end_point = parking_coords[rear_end_idx]
    dists_from_rear = np.linalg.norm(parking_coords - rear_end_point, axis=1)
    close_indices = np.where(dists_from_rear <= movement_threshold)[0]
    candidate_frames = parking_df.iloc[close_indices]['frame']
    adjusted_rear_idx = close_indices[np.argmin(candidate_frames)]
    rear_end_point = parking_coords[adjusted_rear_idx]

    interpolated_point = front_end_point*(1-end_entry_ratio) + rear_end_point*end_entry_ratio

    dists_interpolated = np.linalg.norm(parking_coords - interpolated_point, axis=1)
    final_end_idx = np.argmin(dists_interpolated)
    final_end_point = parking_coords[final_end_idx]
    final_end_frame = parking_df.iloc[final_end_idx]['frame']

    # --- Start-of-Maneuver Detection ---
    peak_idx = None
    consecutive_post_peak_frames=0
    for idx in range(tA, len(driving_coords)-tA):
        r1 = driving_coords[idx] - driving_coords[idx-tA]
        r2 = driving_coords[idx+tA] - driving_coords[idx]
        if is_reversal(r1, r2):
            true_reversal = True
            for t in range(1, tA):
                rA = driving_coords[idx] - driving_coords[max(0, idx-tA-t)]
                rB = driving_coords[min(len(driving_coords)-1, idx+tA+t)] - driving_coords[idx]
                if not is_reversal(rA, rB):
                    true_reversal = False
                    break
            if true_reversal and in_adjacent_zones(front_end_point, driving_coords[idx], loaded_lines):      
                peak_idx = idx    
        if peak_idx is not None and not is_reversal(r1, r2):
            consecutive_post_peak_frames+=1
        else:
            consecutive_post_peak_frames=0
        if consecutive_post_peak_frames>=tA:
            break

    if(peak_idx is None):
        consecutive_turn_frames = 0
        for idx in range(len(driving_coords)-1, 0, -1):
            aspect_ratio1 = driving_df.iloc[idx-1]['height']/driving_df.iloc[idx-1]['width']
            aspect_ratio2 = driving_df.iloc[idx]['height']/driving_df.iloc[idx]['width']
            if(round(aspect_ratio1, 2)!=round(aspect_ratio2, 2)):
                consecutive_turn_frames+=1
            if(consecutive_turn_frames==tA):
                peak_idx = idx

    if on_border(driving_df.iloc[peak_idx]):
        for idx in range(peak_idx, len(driving_df)):
            if not on_border(driving_df.iloc[idx]):
                peak_idx = idx
                break
    
    peak_point = driving_coords[peak_idx]
    peak_frame = driving_df.iloc[peak_idx]['frame']

    bottom_peak_point = peak_point.copy()
    bottom_peak_point[1]+=driving_df.iloc[peak_idx]['height']//2

    threshold_distance = find_shortest_distance(bottom_peak_point, loaded_lines)
    print(f'threshold_distance: {threshold_distance}')
    zone_based_start_idx = 0
    accumulated_distance = 0
    for i in range(peak_idx - 1, -1, -1):
        if threshold_distance is None:
            break

        accumulated_distance+=np.linalg.norm(driving_coords[i+1]-driving_coords[i])
        if(accumulated_distance>=threshold_distance and
           np.linalg.norm(driving_coords[i]-driving_coords[peak_idx])>=threshold_distance):
            zone_based_start_idx = i
            break     

    zone_based_start_frame = driving_df.iloc[zone_based_start_idx]["frame"]
    zone_based_start_point = driving_coords[zone_based_start_idx]

if(maneuver_type==ManeuverType.EXT):
    # detect start of exit maneuver 
    front_start_point = driving_coords[0]
    dists_from_front = np.linalg.norm(parking_coords - front_start_point, axis=1)

    rear_start_idx = np.argmax(dists_from_front)
    rear_start_point = parking_coords[rear_start_idx]

    dists_from_rear = np.linalg.norm(parking_coords - rear_start_point, axis=1)
    close_indices = np.where(dists_from_rear <= movement_threshold)[0]
    candidate_frames = parking_df.iloc[close_indices]['frame']
    adjusted_rear_idx = close_indices[np.argmax(candidate_frames)]
    rear_start_point = parking_coords[adjusted_rear_idx]

    interpolated = front_start_point*(start_exit_ratio) + rear_start_point*(1-start_exit_ratio)

    dists_interpolated = np.linalg.norm(parking_coords - interpolated, axis=1)
    final_start_idx = np.argmin(dists_interpolated)
    final_start_point = parking_coords[final_start_idx]
    final_start_frame = parking_df.iloc[final_start_idx]['frame']

    # detect end of exit maneuver
    peak_idx = None
    consecutive_post_peak_frames=0
    for idx in range(tA, len(driving_coords)-tA):
        r1 = driving_coords[idx] - driving_coords[idx-tA]
        r2 = driving_coords[idx+tA] - driving_coords[idx]
        if is_reversal(r1, r2):
            true_reversal = True
            for t in range(1, tA):
                rA = driving_coords[idx] - driving_coords[max(0, idx-tA-t)]
                rB = driving_coords[min(len(driving_coords)-1, idx+tA+t)] - driving_coords[idx]
                if not is_reversal(rA, rB):
                    true_reversal = False
                    break
            if true_reversal and in_adjacent_zones(front_start_point, driving_coords[idx], loaded_lines):      
                peak_idx = idx    
        if peak_idx is not None and not is_reversal(r1, r2):
            consecutive_post_peak_frames+=1
        else:
            consecutive_post_peak_frames=0
        if consecutive_post_peak_frames>=tA:
            break

    if(peak_idx is None):
        peak_idx = 0
        consecutive_turn_frames = 0
        for idx in range(len(driving_coords)-1):
            aspect_ratio1 = driving_df.iloc[idx]['height']/driving_df.iloc[idx]['width']
            aspect_ratio2 = driving_df.iloc[idx+1]['height']/driving_df.iloc[idx+1]['width']
            if(round(aspect_ratio1, 2)!=round(aspect_ratio2, 2)):
                consecutive_turn_frames+=1
            if(consecutive_turn_frames==tA):
                peak_idx = idx

    if on_border(driving_df.iloc[peak_idx]):
        for idx in range(peak_idx-1, -1, -1):
            if not on_border(driving_df.iloc[idx]):
                peak_idx = idx
                break
    
    peak_point = driving_coords[peak_idx]
    peak_frame = driving_df.iloc[peak_idx]['frame']
    
    bottom_peak_point = peak_point.copy()
    bottom_peak_point[1]+=driving_df.iloc[peak_idx]['height']//2

    threshold_distance = find_shortest_distance(bottom_peak_point, loaded_lines)
    
    zone_based_end_idx = -1
    accumulated_distance = 0
    for i in range(peak_idx, len(driving_coords)-1):
        accumulated_distance+=np.linalg.norm(driving_coords[i+1]-driving_coords[i])
        if(accumulated_distance>=threshold_distance and
           np.linalg.norm(driving_coords[i+1]-driving_coords[peak_idx])>=threshold_distance):
            zone_based_end_idx = i+1
            break

    zone_based_end_frame = driving_df.iloc[zone_based_end_idx]["frame"]
    zone_based_end_point = driving_coords[zone_based_end_idx]

    #print(f'{(zone_based_end_frame - final_start_frame)/30.0}s')
    print(final_start_frame)
    print(zone_based_end_frame)
            
# Base figure
base_fig = go.Figure()
base_fig.add_trace(go.Scatter(
    x=vehicle_df["x"],
    y=vehicle_df["y"],
    mode='lines+markers',
    name='Full Trajectory',
    line=dict(color='lightgray'),
    customdata=np.stack([
        vehicle_df["height"] / (vehicle_df["width"] + 1e-6)  # avoid divide by zero
    ], axis=-1),
    hovertemplate=(
        "X: %{x:.2f}<br>" +
        "Y: %{y:.2f}<br>" +
        "H/W Ratio: %{customdata[0]:.2f}<extra></extra>"
    )
))
start_row = vehicle_df.iloc[(vehicle_df['frame'] - start_frame).abs().argsort().iloc[0]]
base_fig.add_trace(go.Scatter(
    x=[start_row['x']],
    y=[start_row['y']],
    mode='markers',
    marker=dict(color='green', size=12),
    name='Start (annotated)'
))
end_row = vehicle_df.iloc[(vehicle_df['frame'] - end_frame).abs().argsort().iloc[0]]
base_fig.add_trace(go.Scatter(
    x=[end_row['x']],
    y=[end_row['y']],
    mode='markers',
    marker=dict(color='red', size=12),
    name='End (annotated)'
))
base_fig.add_trace(go.Scatter(
    x=parking_df["x"],
    y=parking_df["y"],
    mode='markers',
    marker=dict(color='orange', size=8),
    name='In Parking Zone'
))

if(maneuver_type==ManeuverType.ENT):
    base_fig.add_trace(go.Scatter(
            x=[peak_point[0]],
            y=[peak_point[1]],
            mode='markers',
            marker=dict(color='orange', size=14, symbol='diamond'),
            name='Peak'
        ))

    if zone_based_start_point is not None:
        base_fig.add_trace(go.Scatter(
            x=[zone_based_start_point[0]],
            y=[zone_based_start_point[1]],
            mode='markers',
            marker=dict(color='blue', size=14, symbol='x'),
            name='Zone-based start (trajectory-derived)'
        ))

    if final_end_point is not None: 
        base_fig.add_trace(go.Scatter(
            x=[final_end_point[0]],
            y=[final_end_point[1]],
            mode='markers',
            marker=dict(color='red', size=14, symbol='x'),
            name='End point'
        ))

if(maneuver_type==ManeuverType.EXT):
    base_fig.add_trace(go.Scatter(
            x=[peak_point[0]],
            y=[peak_point[1]],
            mode='markers',
            marker=dict(color='orange', size=14, symbol='diamond'),
            name='Peak'
        ))
    
    if final_start_point is not None:
        base_fig.add_trace(go.Scatter(
            x=[final_start_point[0]],
            y=[final_start_point[1]],
            mode='markers',
            marker=dict(color='green', size=14, symbol='x'),
            name='Start point (automatically determined)'
        ))

    if zone_based_end_point is not None:
        base_fig.add_trace(go.Scatter(
            x=[zone_based_end_point[0]],
            y=[zone_based_end_point[1]],
            mode='markers',
            marker=dict(color='blue', size=14, symbol='diamond'),
            name='Zone-based Exit End Point'
        ))
    

base_fig.update_layout(
    xaxis_title="X Position",
    yaxis_title="Y Position",
    yaxis=dict(autorange='reversed'),
    clickmode='event+select'
)

# App
app = Dash(__name__)
app.title = "Vehicle Trajectory Inspector"

app.layout = html.Div([
    html.H1("Trajectory Inspector"),
    dcc.Graph(
        id="trajectory-plot",
        figure=base_fig,
        config={
            "scrollZoom": True,
            "displayModeBar": True,
            "doubleClick": "reset"
        }
    ),
    dcc.Store(id="clicked-points", data=[]),
    html.Div(id="output", style={"whiteSpace": "pre-line", "marginTop": "20px", "fontSize": "16px"}),
])

if __name__ == "__main__":
    app.run(debug=True)

99.0
399.0


In [3]:
import cv2
import numpy as np

# Open the video file
video_path = f'clips/Parking-clip{video_ID}.mp4'  # Adjust if your video has a different extension
cap = cv2.VideoCapture(video_path)
FPS = vid.get(cv2.CAP_PROP_FPS)
duration_frames = int(0.5*FPS)

# Create a window
cv2.namedWindow('Parking Maneuver Detection', cv2.WINDOW_NORMAL)

# Frame counter
current_frame = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    current_frame += 1
    
    if maneuver_type == ManeuverType.ENT:
        # Check if current frame matches any of our key frames
        if zone_based_start_frame <= current_frame <= zone_based_start_frame + duration_frames:
            # Tint blue for zone-based start frame
            tint = np.zeros_like(frame)
            tint[:] = (255, 0, 0)  # Blue in BGR
            frame = cv2.addWeighted(frame, 0.7, tint, 0.3, 0)
            text = "Zone-Based Start"
        elif peak_frame <= current_frame <= peak_frame + duration_frames:
            # Tint orange for peak frame
            tint = np.zeros_like(frame)
            tint[:] = (0, 165, 255)  # Orange in BGR
            frame = cv2.addWeighted(frame, 0.7, tint, 0.3, 0)
            text = "Peak Frame"
        elif start_frame <= current_frame <= start_frame + duration_frames:
            # Tint green annotated start frame
            tint = np.zeros_like(frame)
            tint[:] = (0, 255, 0)  # Green in BGR
            frame = cv2.addWeighted(frame, 0.7, tint, 0.3, 0)
            text = "Annotated Start Frame"
        elif final_end_frame <= current_frame <= final_end_frame + duration_frames:
            # Tint red for end frame
            tint = np.zeros_like(frame)
            tint[:] = (0, 0, 255)  # Red in BGR
            frame = cv2.addWeighted(frame, 0.7, tint, 0.3, 0)
            text = "End Frame"
        else:
            text = None

    if maneuver_type == ManeuverType.EXT:
        # Check if current frame matches any of our key frames
        if zone_based_end_frame <= current_frame <= zone_based_end_frame + duration_frames:
            # Tint blue for zone-based start frame
            tint = np.zeros_like(frame)
            tint[:] = (255, 0, 0)  # Blue in BGR
            frame = cv2.addWeighted(frame, 0.7, tint, 0.3, 0)
            text = "Zone-Based End"
        elif peak_frame <= current_frame <= peak_frame + duration_frames:
            # Tint orange for peak frame
            tint = np.zeros_like(frame)
            tint[:] = (0, 165, 255)  # Orange in BGR
            frame = cv2.addWeighted(frame, 0.7, tint, 0.3, 0)
            text = "Peak Frame"
        elif end_frame <= current_frame <= end_frame + duration_frames:
            # Tint red for annotated end frame
            tint = np.zeros_like(frame)
            tint[:] = (0, 0, 255)  # Red in BGR
            frame = cv2.addWeighted(frame, 0.7, tint, 0.3, 0)
            text = "Annotated End Frame"
        elif final_start_frame <= current_frame <= final_start_frame + duration_frames:
            # Tint green for start frame
            tint = np.zeros_like(frame)
            tint[:] = (0, 255, 0)  # Green in BGR
            frame = cv2.addWeighted(frame, 0.7, tint, 0.3, 0)
            text = "Start Frame"
        else:
            text = None

    # Display frame number
    cv2.putText(frame, f"Frame: {current_frame}", (10, 30), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    
    # Display text for key frames
    if text:
        cv2.putText(frame, text, (10, 60), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    
    # Show the frame
    cv2.imshow('Parking Maneuver Detection', frame)
    
    # Slow down playback for visualization
    key = cv2.waitKey(50)  # ~20 fps playback
    
    # Exit if 'q' is pressed
    if key == ord('q'):
        break

# Release resources
cap.release()
cv2.destroyAllWindows()

In [2]:
import cv2
import dash
from dash import dcc, html, Input, Output
import dash_bootstrap_components as dbc
import numpy as np
import pandas as pd
import plotly.graph_objs as go
import threading
import time
import base64
from collections import deque
from enum import Enum
import json

video_ID = '1ENT'

# Enter hyperparameters (adjustable)
start_entry_ratio = 0.5 # 0 = zone based | 1 = peak based 
end_entry_ratio = 1.0 # 0 = front parking zone | 1 = rear parking zone

# Exit hyperparameters (adjustable)
start_exit_ratio = 0 # 0 = rear parking zone | 1 = front parking zone
end_exit_ratio = 1.0 # 0 = peak based | 1 = zone based

# low motion hyperparameters 
movement_threshold = 1.0
rotation_threshold = 0.01
rotation_lockout = 0.1 

# Peak hyperparameters 
tA = 5
aA = np.pi/6

video_path = f'/home/edisonz/maneuver_heuristics25/clips/Parking-clip{video_ID}.mp4'

vid = cv2.VideoCapture(video_path)
frame_width = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))  # Max x-coordinate
frame_height = 2 * int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT)) // 3  # Max y-coordinate
FPS = vid.get(cv2.CAP_PROP_FPS)
if FPS == 0 or np.isnan(FPS):
    FPS = 30  # fallback

# Duration frames to highlight start/end (you can adjust)
duration_frames = int(FPS * 1)  # 1 second duration highlight

# Load data
vehicle_df = pd.read_csv(f'clips/Parking-clip{video_ID}.csv')
maneuvers = pd.read_csv(f'clip-annotations/maneuver{video_ID}.csv')
start_frame, end_frame = np.median(maneuvers["start_frame"]), np.median(maneuvers["end_frame"])
driving_df = vehicle_df[vehicle_df["in_parking_zone"]==0]
driving_coords = driving_df[['x', 'y']].values
parking_df = vehicle_df[vehicle_df["in_parking_zone"]==1]
parking_coords = parking_df[['x', 'y']].values
start_frame = int(round(start_frame))
end_frame = int(round(end_frame))
parking_lines = 'lines.json'

frame_numbers = vehicle_df['frame'].values
trajectory = vehicle_df[['x', 'y']].values

frame_state = {'index': 0, 'image': None}
frame_lock = threading.Lock()

class ManeuverType(Enum):
    ENT = 0
    EXT = 1

maneuver_type = ManeuverType.ENT if video_ID[-3:].lower()=='ent' else ManeuverType.EXT

video_ID = '1ENT'

vid = cv2.VideoCapture(f'/home/edisonz/maneuver_heuristics25/Parking-clip{video_ID}.mp4')
frame_width = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))  # Max x-coordinate
frame_height = 2*int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))//3  # Max y-coordinate
# ----------------------------------------------------------------------------------------------------------------------------

# Enter hyperparameters (adjustable)
start_entry_ratio = 0.5 # 0 = zone based | 1 = peak based 
end_entry_ratio = 1.0 # 0 = front parking zone | 1 = rear parking zone

# Exit hyperparameters (adjustable)
start_exit_ratio = 0 # 0 = rear parking zone | 1 = front parking zone
end_exit_ratio = 1.0 # 0 = peak based | 1 = zone based

# low motion hyperparameters 
movement_threshold = 1.0

# Peak hyperparameters 
tA = 5
aA = np.pi/6

def load_lines(json_file):
    with open(json_file, 'r') as f:
        data = json.load(f)
    lines = [list(map(tuple, line)) for line in data['lines']]
    loaded_lines = []
    for line in lines: 
        m = (line[1][1] - line[0][1])/(line[1][0] - line[0][0])
        c = line[0][1] - m * line[0][0]
        loaded_lines.append((m, c))
    return loaded_lines

# Load data
vehicle_df = pd.read_csv(f'clips/Parking-clip{video_ID}.csv')
maneuvers = pd.read_csv(f'clip-annotations/maneuver{video_ID}.csv')
start_frame, end_frame = np.median(maneuvers["start_frame"]), np.median(maneuvers["end_frame"])
driving_df = vehicle_df[vehicle_df["in_parking_zone"]==0]
driving_coords = driving_df[['x', 'y']].values
parking_df = vehicle_df[vehicle_df["in_parking_zone"]==1]
parking_coords = parking_df[['x', 'y']].values
start_frame = int(round(start_frame))
end_frame = int(round(end_frame))

region_code = None
with open("mappings.json", 'r') as f:
    data = json.load(f)
    region_code = data.get(f'{video_ID}')
parking_lines = f'lines{region_code}.json'

loaded_lines = load_lines(parking_lines)

class ManeuverType(Enum):
    ENT = 0
    EXT = 1

maneuver_type = ManeuverType.ENT if video_ID[-3:].lower()=='ent' else ManeuverType.EXT

# --------------------------------------------------------------------------------------------------------------------------

# Determine if direction reversal in x or y occured
def is_reversal(r1, r2):
    angle_diff = np.abs(np.arctan2(r1[1], r1[0]) - np.arctan2(r2[1], r2[0]))
    if angle_diff < aA:
        return False  # No significant angle change
    return (r1[0] * r2[0] < 0) or (r1[1] * r2[1] < 0)  # Opposite signs in x or y

# Determine if vehicle is hugging the border of the frame
def on_border(df_row):
    height = df_row["height"]
    width = df_row["width"]
    x = df_row["x"]
    y = df_row["y"]
    left_x = int(x-width//2)
    right_x = int(x+width//2)
    top_y = int(y-height//2)
    bottom_y = int(y+height//2)
    return (left_x==0 or top_y==0 or right_x==frame_width or bottom_y==frame_height)

def find_shortest_distance(point, loaded_lines):
    """
    Compute the shortest distance for a given point:
    - If the point lies between two parking lines, return the sum of perpendicular distances to the nearest lines on either side.
    - If the point does not lie between two lines, return the perpendicular distance to the closest line (min perp dist among all lines).
    Lines include both original from json and extrapolated ones.
    
    Args:
        point (tuple): (x, y) coordinates of the point
        json_file (str): Path to the JSON file containing line data
    
    Returns:
        float: Shortest distance (sum of perpendicular distances or min distance to closest line), or None if no lines exist
    """
    x_point, y_point = point

    # Original lines
    intersections = []
    line_params = []
    for m, c in loaded_lines:
        x_inter = (y_point - c) / m
        intersections.append(x_inter)

    # Separate intersections into left and right
    left = [(x, params) for x, params in zip(intersections, loaded_lines) if x < x_point]
    right = [(x, params) for x, params in zip(intersections, loaded_lines) if x > x_point]

    # Compute perpendicular distance to a line
    def point_to_line_distance(point, line_params):
        x_p, y_p = point
        if line_params[1] is None:  # Vertical line: x = c
            return abs(x_p - line_params[0])
        m, c = line_params
        return abs(m * x_p - y_p + c) / np.sqrt(m**2 + 1)

    # Case 1: Point lies between two lines (considering all)
    if left and right:
        __, left_params = max(left, key=lambda x: x[0])
        __, right_params = min(right, key=lambda x: x[0])
        d_left = point_to_line_distance(point, left_params)
        d_right = point_to_line_distance(point, right_params)
        return d_left + d_right

    # Case 2: Point does not lie between two lines, find closest line by min perp dist
    distances = [point_to_line_distance(point, params) for params in line_params]
    return min(distances)

def in_adjacent_zones(p1, p2, loaded_lines):
    """This function calculates whether two points are in the same or adjacent zones"""
    x_point1 = p1[0]
    y_point1 = p1[1]
    x_point2 = p2[0]
    y_point2 = p2[1]

    intersections1 = []
    intersections2 = []
    for m, c in loaded_lines:
        x_inter1 = (y_point1 - c) / m
        intersections1.append(x_inter1)
        x_inter2 = (y_point2 - c) / m
        intersections2.append(x_inter2)

    left1 = [(x, params) for x, params in zip(intersections1, loaded_lines) if x < x_point1]
    right1 = [(x, params) for x, params in zip(intersections1, loaded_lines) if x > x_point1]

    left2 = [(x, params) for x, params in zip(intersections2, loaded_lines) if x < x_point2]
    right2 = [(x, params) for x, params in zip(intersections2, loaded_lines) if x > x_point2]

    if len(left1) > 0: 
        __, left_params1 = max(left1, key = lambda x: x[0])  
    else: 
        left_params1 = None

    if len(right1) > 0:
        __, right_params1 = min(right1, key = lambda x: x[0]) 
    else:
        right_params1 = None

    if len(left2) > 0: 
        __, left_params2 = max(left2, key = lambda x: x[0])  
    else: 
        left_params2 = None

    if len(right2) > 0:
        __, right_params2 = min(right2, key = lambda x: x[0]) 
    else:
        right_params2 = None

    return (left_params1 == left_params2 or left_params1 == right_params2 or 
            right_params1 == right_params2 or right_params1 == left_params2)

if maneuver_type == ManeuverType.ENT:
    # --- End-of-Maneuver Detection (STOP) ---
    front_end_point = parking_coords[0]
    dists_from_front = np.linalg.norm(parking_coords - front_end_point, axis=1)

    rear_end_idx = np.argmax(dists_from_front)
    rear_end_point = parking_coords[rear_end_idx]
    dists_from_rear = np.linalg.norm(parking_coords - rear_end_point, axis=1)
    close_indices = np.where(dists_from_rear <= movement_threshold)[0]
    candidate_frames = parking_df.iloc[close_indices]['frame']
    adjusted_rear_idx = close_indices[np.argmin(candidate_frames)]
    rear_end_point = parking_coords[adjusted_rear_idx]

    interpolated_point = front_end_point*(1-end_entry_ratio) + rear_end_point*end_entry_ratio

    dists_interpolated = np.linalg.norm(parking_coords - interpolated_point, axis=1)
    final_end_idx = np.argmin(dists_interpolated)
    final_end_point = parking_coords[final_end_idx]
    final_end_frame = parking_df.iloc[final_end_idx]['frame']

    # --- Start-of-Maneuver Detection ---
    peak_idx = None
    consecutive_post_peak_frames=0
    for idx in range(tA, len(driving_coords)-tA):
        r1 = driving_coords[idx] - driving_coords[idx-tA]
        r2 = driving_coords[idx+tA] - driving_coords[idx]
        if is_reversal(r1, r2):
            true_reversal = True
            for t in range(1, tA):
                rA = driving_coords[idx] - driving_coords[max(0, idx-tA-t)]
                rB = driving_coords[min(len(driving_coords)-1, idx+tA+t)] - driving_coords[idx]
                if not is_reversal(rA, rB):
                    true_reversal = False
                    break
            if true_reversal and in_adjacent_zones(front_end_point, driving_coords[idx], loaded_lines):      
                peak_idx = idx    
        if peak_idx is not None and not is_reversal(r1, r2):
            consecutive_post_peak_frames+=1
        else:
            consecutive_post_peak_frames=0
        if consecutive_post_peak_frames>=tA:
            break

    if(peak_idx is None):
        consecutive_turn_frames = 0
        for idx in range(len(driving_coords)-1, 0, -1):
            aspect_ratio1 = driving_df.iloc[idx-1]['height']/driving_df.iloc[idx-1]['width']
            aspect_ratio2 = driving_df.iloc[idx]['height']/driving_df.iloc[idx]['width']
            if(round(aspect_ratio1, 2)!=round(aspect_ratio2, 2)):
                consecutive_turn_frames+=1
            if(consecutive_turn_frames==tA):
                peak_idx = idx

    if on_border(driving_df.iloc[peak_idx]):
        for idx in range(peak_idx, len(driving_df)):
            if not on_border(driving_df.iloc[idx]):
                peak_idx = idx
                break
    
    peak_point = driving_coords[peak_idx]
    peak_frame = driving_df.iloc[peak_idx]['frame']

    bottom_peak_point = peak_point.copy()
    bottom_peak_point[1]+=driving_df.iloc[peak_idx]['height']//2

    threshold_distance = find_shortest_distance(bottom_peak_point, loaded_lines)
    print(f'threshold_distance: {threshold_distance}')
    zone_based_start_idx = 0
    accumulated_distance = 0
    for i in range(peak_idx - 1, -1, -1):
        if threshold_distance is None:
            break

        accumulated_distance+=np.linalg.norm(driving_coords[i+1]-driving_coords[i])
        if(accumulated_distance>=threshold_distance and
           np.linalg.norm(driving_coords[i]-driving_coords[peak_idx])>=threshold_distance):
            zone_based_start_idx = i
            break     

    zone_based_start_frame = driving_df.iloc[zone_based_start_idx]["frame"]
    zone_based_start_point = driving_coords[zone_based_start_idx]

if(maneuver_type==ManeuverType.EXT):
    # detect start of exit maneuver 
    front_start_point = driving_coords[0]
    dists_from_front = np.linalg.norm(parking_coords - front_start_point, axis=1)

    rear_start_idx = np.argmax(dists_from_front)
    rear_start_point = parking_coords[rear_start_idx]

    dists_from_rear = np.linalg.norm(parking_coords - rear_start_point, axis=1)
    close_indices = np.where(dists_from_rear <= movement_threshold)[0]
    candidate_frames = parking_df.iloc[close_indices]['frame']
    adjusted_rear_idx = close_indices[np.argmax(candidate_frames)]
    rear_start_point = parking_coords[adjusted_rear_idx]

    interpolated = front_start_point*(start_exit_ratio) + rear_start_point*(1-start_exit_ratio)

    dists_interpolated = np.linalg.norm(parking_coords - interpolated, axis=1)
    final_start_idx = np.argmin(dists_interpolated)
    final_start_point = parking_coords[final_start_idx]
    final_start_frame = parking_df.iloc[final_start_idx]['frame']

    # detect end of exit maneuver
    peak_idx = None
    consecutive_post_peak_frames=0
    for idx in range(tA, len(driving_coords)-tA):
        r1 = driving_coords[idx] - driving_coords[idx-tA]
        r2 = driving_coords[idx+tA] - driving_coords[idx]
        if is_reversal(r1, r2):
            true_reversal = True
            for t in range(1, tA):
                rA = driving_coords[idx] - driving_coords[max(0, idx-tA-t)]
                rB = driving_coords[min(len(driving_coords)-1, idx+tA+t)] - driving_coords[idx]
                if not is_reversal(rA, rB):
                    true_reversal = False
                    break
            if true_reversal and in_adjacent_zones(front_start_point, driving_coords[idx], loaded_lines):      
                peak_idx = idx    
        if peak_idx is not None and not is_reversal(r1, r2):
            consecutive_post_peak_frames+=1
        else:
            consecutive_post_peak_frames=0
        if consecutive_post_peak_frames>=tA:
            break

    if(peak_idx is None):
        peak_idx = 0
        consecutive_turn_frames = 0
        for idx in range(len(driving_coords)-1):
            aspect_ratio1 = driving_df.iloc[idx]['height']/driving_df.iloc[idx]['width']
            aspect_ratio2 = driving_df.iloc[idx+1]['height']/driving_df.iloc[idx+1]['width']
            if(round(aspect_ratio1, 2)!=round(aspect_ratio2, 2)):
                consecutive_turn_frames+=1
            if(consecutive_turn_frames==tA):
                peak_idx = idx

    if on_border(driving_df.iloc[peak_idx]):
        for idx in range(peak_idx-1, -1, -1):
            if not on_border(driving_df.iloc[idx]):
                peak_idx = idx
                break
    
    peak_point = driving_coords[peak_idx]
    peak_frame = driving_df.iloc[peak_idx]['frame']
    
    bottom_peak_point = peak_point.copy()
    bottom_peak_point[1]+=driving_df.iloc[peak_idx]['height']//2

    threshold_distance = find_shortest_distance(bottom_peak_point, loaded_lines)
    
    zone_based_end_idx = -1
    accumulated_distance = 0
    for i in range(peak_idx, len(driving_coords)-1):
        accumulated_distance+=np.linalg.norm(driving_coords[i+1]-driving_coords[i])
        if(accumulated_distance>=threshold_distance and
           np.linalg.norm(driving_coords[i+1]-driving_coords[peak_idx])>=threshold_distance):
            zone_based_end_idx = i+1
            break

    zone_based_end_frame = driving_df.iloc[zone_based_end_idx]["frame"]
    zone_based_end_point = driving_coords[zone_based_end_idx]

# --- Video thread management ---

video_thread_running = False

def video_reader():
    global video_thread_running
    cap = cv2.VideoCapture(video_path)
    frame_number = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            frame_number = 0
            continue

        with frame_lock:
            frame_state['index'] = frame_number

            overlay = None
            text = None
            if maneuver_type==ManeuverType.EXT:
                if final_start_frame <= frame_number <= final_start_frame + duration_frames:
                    overlay = (0, 255, 0)
                    text = "Detected Start Frame"
                elif end_frame <= frame_number <= end_frame + duration_frames:
                    overlay = (0, 0, 255)
                    text = "Annotated End Frame"
                elif peak_frame <= frame_number <= peak_frame + duration_frames:
                    overlay = (0, 165, 255)
                    text = "Peak Frame"
                elif zone_based_end_frame <= frame_number <= zone_based_end_frame + duration_frames:
                    overlay = (255, 0, 0)
                    text = "Zone Based Start Frame"
            if maneuver_type==ManeuverType.ENT:
                if start_frame <= frame_number <= start_frame + duration_frames:
                    overlay = (0, 255, 0)
                    text = "Annotated Start Frame"
                elif final_end_frame <= frame_number <= final_end_frame + duration_frames:
                    overlay = (0, 0, 255)
                    text = "Detected End Frame"
                elif peak_frame <= frame_number <= peak_frame + duration_frames:
                    overlay = (0, 165, 255)
                    text = "Peak Frame"
                elif zone_based_start_frame <= frame_number <= zone_based_start_frame + duration_frames:
                    overlay = (255, 0, 0)
                    text = "Zone Based End Frame"

            if overlay is not None:
                overlay_img = np.full_like(frame, overlay)
                frame = cv2.addWeighted(frame, 0.7, overlay_img, 0.3, 0)

            cv2.putText(frame, f"Frame: {frame_number}", (10, 120),
                        cv2.FONT_HERSHEY_SIMPLEX, 3.5, (255, 255, 255), 2)
            if text:
                cv2.putText(frame, text, (10, 290),  # Adjusted Y position to avoid overlap
                            cv2.FONT_HERSHEY_SIMPLEX, 3.5, (255, 255, 255), 2)

            _, buffer = cv2.imencode('.jpg', frame)
            jpg_as_text = base64.b64encode(buffer).decode()
            frame_state['image'] = f"data:image/jpeg;base64,{jpg_as_text}"

        frame_number += 1
        time.sleep(1 / FPS)

    cap.release()
    video_thread_running = False

def start_video_reader():
    global video_thread_running
    if not video_thread_running:
        video_thread_running = True
        threading.Thread(target=video_reader, daemon=True).start()

# --- Plot base figure once ---

def create_base_figure():
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=vehicle_df['x'], y=vehicle_df['y'],
        mode='lines', name='Trajectory', line=dict(color='gray')
    ))

    start_row = vehicle_df.iloc[(vehicle_df['frame'] - start_frame).abs().argsort().iloc[0]]
    end_row = vehicle_df.iloc[(vehicle_df['frame'] - end_frame).abs().argsort().iloc[0]]

    fig.add_trace(go.Scatter(
        x=[start_row['x']], y=[start_row['y']],
        mode='markers', marker=dict(color='green', size=12),
        name='Start (annotated)'
    ))
    fig.add_trace(go.Scatter(
        x=[end_row['x']], y=[end_row['y']],
        mode='markers', marker=dict(color='red', size=12),
        name='End (annotated)'
    ))

    fig.add_trace(go.Scatter(
        x=parking_df['x'], y=parking_df['y'],
        mode='markers', marker=dict(color='orange', size=8),
        name='In Parking Zone'
    ))

    if maneuver_type == ManeuverType.ENT:
        fig.add_trace(go.Scatter(
            x=[peak_point[0]], y=[peak_point[1]],
            mode='markers', marker=dict(color='orange', size=14, symbol='diamond'),
            name='Peak'
        ))
        if zone_based_start_point is not None:
            fig.add_trace(go.Scatter(
                x=[zone_based_start_point[0]], y=[zone_based_start_point[1]],
                mode='markers', marker=dict(color='blue', size=14, symbol='x'),
                name='Zone-based start (trajectory-derived)'
            ))
        if final_end_point is not None:
            fig.add_trace(go.Scatter(
                x=[final_end_point[0]], y=[final_end_point[1]],
                mode='markers', marker=dict(color='red', size=14, symbol='x'),
                name='End point'
            ))

    if maneuver_type == ManeuverType.EXT:
        fig.add_trace(go.Scatter(
            x=[peak_point[0]], y=[peak_point[1]],
            mode='markers', marker=dict(color='orange', size=14, symbol='diamond'),
            name='Peak'
        ))
        if final_start_point is not None:
            fig.add_trace(go.Scatter(
                x=[final_start_point[0]], y=[final_start_point[1]],
                mode='markers', marker=dict(color='green', size=14, symbol='x'),
                name='Start point (automatically determined)'
            ))
        if zone_based_end_point is not None:
            fig.add_trace(go.Scatter(
                x=[zone_based_end_point[0]], y=[zone_based_end_point[1]],
                mode='markers', marker=dict(color='blue', size=14, symbol='diamond'),
                name='Zone-based Exit End Point'
            ))

    fig.update_layout(
        xaxis_title='X',
        yaxis_title='Y',
        yaxis=dict(autorange='reversed'),
        height=500,
    )

    return fig

base_fig = create_base_figure()

# --- Dash app setup ---

app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])
app.title = "Live Vehicle Tracking + Trajectory"

app.layout = dbc.Container([
    html.H3("Live Vehicle Tracking"),
    dbc.Row([
        dbc.Col(html.Img(id='video-frame', style={'width': '100%'}), width=6),
        dbc.Col(dcc.Graph(id='trajectory-plot'), width=6),
    ]),
    dcc.Interval(id='update', interval=100, n_intervals=0),  # 10 FPS update
], fluid=True)

start_video_reader()  # Start video thread once here

@app.callback(
    Output('video-frame', 'src'),
    Output('trajectory-plot', 'figure'),
    Input('update', 'n_intervals')
)
def update_ui(n):
    with frame_lock:
        image_data = frame_state.get('image', None)
        cf = frame_state.get('index', 0)

    if image_data is None:
        return dash.no_update, dash.no_update

    if cf >= frame_numbers[-1]:
        cf = frame_numbers[-1]

    closest_idx = np.argmin(np.abs(frame_numbers - cf))
    curr_x, curr_y = trajectory[closest_idx]

    fig = go.Figure(base_fig)  # copy base figure

    fig.add_trace(go.Scatter(
        x=[curr_x], y=[curr_y],
        mode='markers+text',
        marker=dict(size=12, color='red'),
        name='Now',
        text=['Now'],
        textposition="top center"
    ))

    return image_data, fig

if __name__ == '__main__':
    app.run(debug=True)


threshold_distance: 93.43766401419732
